# PoC de reunião com LangChain 

## 0. Instalando dependências

In [9]:
%pip install -q python-dotenv pydantic langchain-core langchain-google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [10]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

## 2. Chave de API

In [11]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

## 3. Leitura da transcrição

O texto da reunião é obtido de `meeting-text.txt` e será enviado ao modelo como contexto.

In [12]:
with open("meeting-text.txt", "r", encoding="utf-8") as meeting_text_file:
    text = meeting_text_file.read()

print("Texto da reunião carregado com sucesso.")

Texto da reunião carregado com sucesso.


## 4. Formato esperado da resposta

O modelo deve retornar um objetivo principal e uma lista de tarefas. O Pydantic valida essa estrutura depois que a resposta é recebida.

In [13]:
class MeetingSummary(BaseModel):
    tasks: list[str] = Field(..., description="Lista de tarefas extraídas da reunião.")

    objective: str = Field(..., description="Objetivo principal extraído da reunião.")


parser = PydanticOutputParser(pydantic_object=MeetingSummary)

print("PydanticOutputParser criado com sucesso")

PydanticOutputParser criado com sucesso


## 5. Modelo e prompt

Configuramos o Gemini com temperatura zero para respostas mais consistentes. O parser inclui no prompt as instruções necessárias para o retorno seguir a estrutura definida acima.

In [14]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0,
)

prompt = PromptTemplate(
    input_variables=["meeting_text"],
    template="""
Você é um assistente de inteligência artificial especializado
em extrair informações de reuniões.

Analise a reunião abaixo e extraia:
o objetivo principal;
as tarefas definidas.

Não invente informações que não estejam presentes na reunião.

Reunião:
{meeting_text}

{format_instructions}
""",
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

print("PromptTemplate criado com sucesso.")

PromptTemplate criado com sucesso.


## 6. Montagem e envio do prompt

Nesta etapa inserimos a transcrição no template e enviamos o prompt completo ao Gemini. A execução desta célula requer uma chave de API válida.

In [15]:
prompt_final = prompt.format(meeting_text=text)

llm_response = llm.invoke(prompt_final)

print("Resposta do modelo recebida com sucesso.")

Resposta do modelo recebida com sucesso.


## 7. Validação e resultado final

Primeiro exibimos a resposta bruta para facilitar a depuração. Em seguida, o parser converte e valida o conteúdo no modelo `MeetingSummary`, e mostramos o resultado como dicionário.

In [16]:
# RESPOSTA BRUTA
print(llm_response.content)

```json
{
  "objective": "Fechar o planejamento da próxima sprint, revisar o que ficou pendente da anterior e dividir as tasks.",
  "tasks": [
    "Maria: Fechar o PR de autenticação e implementar tratamento de sessão expirada.",
    "Maria: Atualizar a documentação (critérios de aceite no ticket e comportamento final na documentação).",
    "Pedro: Fechar o escopo do dashboard e criar as tasks.",
    "Pedro: Enviar os nomes finais das métricas para Bia até sexta de manhã.",
    "Bia: Fazer os componentes iniciais do dashboard e o filtro reutilizável.",
    "Bia: Fazer a estrutura do módulo de usuários usando os mesmos componentes do dashboard.",
    "Bia: Priorizar estados de erro e loading bem definidos nos componentes.",
    "João: Verificar o deploy (testar migration, rollback, health check, comportamento com serviço externo fora do ar e log).",
    "João: Terminar a análise do cache e fechar a solução baseada em números.",
    "João: Melhorar o CI (paralelizar jobs e cache de depe

In [17]:
# Print com resposta final do modelo, já parseada pelo PydanticOutputParser

parsed_output = parser.parse(llm_response.content)
print(parsed_output.model_dump())

{'tasks': ['Maria: Fechar o PR de autenticação e implementar tratamento de sessão expirada.', 'Maria: Atualizar a documentação (critérios de aceite no ticket e comportamento final na documentação).', 'Pedro: Fechar o escopo do dashboard e criar as tasks.', 'Pedro: Enviar os nomes finais das métricas para Bia até sexta de manhã.', 'Bia: Fazer os componentes iniciais do dashboard e o filtro reutilizável.', 'Bia: Fazer a estrutura do módulo de usuários usando os mesmos componentes do dashboard.', 'Bia: Priorizar estados de erro e loading bem definidos nos componentes.', 'João: Verificar o deploy (testar migration, rollback, health check, comportamento com serviço externo fora do ar e log).', 'João: Terminar a análise do cache e fechar a solução baseada em números.', 'João: Melhorar o CI (paralelizar jobs e cache de dependência).', 'João: Criar a documentação de deploy.', 'Lucas: Fechar o desenho da integração e validar o contrato com a API (confirmar uso de cursor e webhooks).', 'Lucas: F